In [0]:
spark.range(10).show()

In [0]:
sample_data = """order_id,customer_name,amount
1,Alice,250
2,Bob,175
3,Carol,300"""

# adjust catalog/schema/volume names to match what you created in Step 1
volume_path = "/Volumes/workspace/default/raw_uploads/sample_orders.csv"

dbutils.fs.put(volume_path, sample_data, overwrite=True)

df = spark.read.csv(volume_path, header=True, inferSchema=True)
df.printSchema()
df.show()

In [0]:
%sql
LIST '/Volumes/workspace/default/raw_uploads'

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/raw_uploads/olist_orders_dataset.csv",
    header=True,
    inferSchema=True,
)
df.printSchema()
print(f"total rows: {df.count()}")
print("total rows (no f):" + str(df.count()))
#df.show(5)



In [0]:
df.display(5)

In [0]:
#DAY 2 - STEP 1:

# #we use orders_df instead of df to avoid confusion in the future with mutiltple tables.
orders_df = spark.read.csv( 
    "/Volumes/workspace/default/raw_uploads/olist_orders_dataset.csv",
    header = True, #csv files are separated by commas, so this tells spark the first row is column names and not actual data
    inferSchema = True, #tells spark to scan through the file and figure out the real type of each column (aka int,text or date etc)
)
orders_df.printSchema()
#this puts all the data into named columns making it easer to use later on. 

In [0]:
#DAY 2 - STEP 2:
orders_df.select("order_id", "order_status").show(10)
#"orderid, orderstatus" taking specific columns in a data set in order to get more specific data, 
# show(10) --> pulls out 10 rows. 

In [0]:

#DAY 2 - STEP 3:

delivered_orders = orders_df.filter(orders_df.order_status == "delivered")
#orders_df.order_status- dot notation for referring to a specific columns inthe dataframe, == "delivered" - a comparioson like english, 
# whole line basically says give me a new dataframe containing only the rows where order status is delivered
delivered_orders.show(5)
print(f"Delivered orders: {delivered_orders.count()}")


In [0]:
#DAY 2 - STEP 4:

from pyspark.sql import functions as F
#imports sparks library of built in funcitons --> gives short nickname to avoid having to write it out everytime
orders_with_date = orders_df.withColumn( # the first argument is the name of new column, the second is how to calculate
    "order_date", F.date_format(F.col("order_purchase_timestamp"), "dd-MM-yyyy")
).withColumn("order_time", F.date_format(F.col("order_purchase_timestamp"), "HH:mm:ss"))
orders_with_date.select("order_id", "order_purchase_timestamp", "order_date", "order_time").show(5)

# this makes reading dates cleaner, raw files timestamp is usually way more presice than needed.


In [0]:
orders_with_timestamp = orders_df.withColumn("order_time", F.date_format(F.col("order_purchase_timestamp"), "HH:mm:ss"))
orders_with_timestamp.select("order_id", "order_time", "order_purchase_timestamp").show(5)

In [0]:
#DAY 2 - STEP 5:

status_summary = orders_df.groupBy("order_status").agg(
    F.count("*").alias("order_count")
    ).orderBy(F.asc("order_count"))
status_summary.show()


#groupBy("order_status")- buckets all rows by orderstatus value (delivered in one, cancelled in another)
#.agg - short for aggregate - calucate smtg for each bucket
#F.count("*") - count every row in each bucket, regardless of column
#.alias("order_count")- renames the coulmn to be readable. otherwise spark would give name like count(1)

#most common operaton in data engineering work - "how mnay per category"


In [0]:
#DAY 2 - STEP 6:
status_summary = status_summary.orderBy(F.desc("order_count"));
status_summary.show()

#F.desc("order_count")- says to sort by this column descending
#F.asc would mean smallest first

#this is used bc raw groups results come out in random order. sorting them makes it more meaningful. 

In [0]:
# cell a - build up a chain of transmortaions. notice: this runs fast
step1 = orders_df.filter(orders_df.order_status == "delivered")
#this isnt filtering data but its creating a recipe decrisbing how to produce the data later on
step2 = step1.withColumn("order_date", F.to_date("order_purchase_timestamp"))
#its adding another instruction to step 1
step3 = step2.groupBy("order_date").agg(F.count("*").alias("daily_count"))

print("Plan Built - no data processed yet.")

In [0]:
#cell b- this is when aprak actually reads the file and does the work
step3.orderBy(F.desc("daily_count")).show(10)

#cell b takes longer to process becasue its taking all the insructions written in cell a and actually processing it. 

In [0]:
#DAY 3 - STEP 1:

customers_df = spark.read.csv(
    "/Volumes/workspace/default/raw_uploads/olist_customers_dataset.csv",
    header = True,
    inferSchema = True,
)
customers_df.printSchema()
customers_df.show(5)

In [0]:
#DAY 3 - STEP 2:

joined_df = orders_df.join(customers_df, on="customer_id", how="inner")
joined_df.select("order_id", "customer_id", "order_status", "customer_state").show(10)
print(f"Joined row count: {joined_df.count()}") 

#this is joining order_df with customers_df to see which colomns have similaries
#inner - means to only keep rows where a math exits in both tables

In [0]:
#DAY 3 - STEP 3:

left_joined_df = orders_df.join(customers_df, on="customer_id", how="left")
print(f"Left joined row count: {left_joined_df.count()}")

#this is saying to keep every single row from order_df and only keep rows from customers_df where a match exists
#left - means to keep all rows from left table and only keep rows from right table where a match exists


In [0]:
#DAY 3 - STEP 4:

missing_customers = orders_df.join(customers_df, on="customer_id", how="left_anti")
print(f"Orders with no matching customer: {missing_customers.count()}")

#this doesnt ombine anything but is asking to show rows from orders_df that have no match in customers_df
#left_anti - means to keep all rows from left table where there is no match in right table


In [0]:
#DAY 3 - STEP 5:
#datalake = your data file (like a csv or parquet fle) + a built in changelong/ safety net

In [0]:
#DAY 3 - STEP 6:
joined_df.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/delta_test/orders_with_customers"
)

#.write = switches from reading data to writing it out somewhere
#.format("delta") = specifies the format of the data being written out
#.mode("overwrite") = specifies the mode of writing out the data, in this case, overwrite means that if the data already exists, it will be overwritten
#.save() = specifies the location where the data will be written out

In [0]:
#DAY 3 - STEP 7:

delta_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/raw_uploads/delta_test/orders_with_customers"
)
delta_df.show(5)
print(f"row count from delta table: {delta_df.count()}")

#confirms the rond trip actually worked 
#this is reading the data back in from the delta table
#this is the same read pattern already known, just swapping .csv for.format("delta").load..

In [0]:

%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/default/raw_uploads/delta_test/orders_with_customers`

--delta.'/path' = special syntax for accessing a delta table directly thru its file path
--DESCRIBE HISTORY = shows history of changes to the table (delta specific command)

In [0]:
#inspecting checkpoint folder:
display(dbutils.fs.ls(f"{CHECKPOINT_PATH}"))

In [0]:
# create one fake "new" batch that wasn't in your original set
import datetime

fake_new_batch = orders_with_week.filter(F.col("order_week") == distinct_weeks[0]).limit(5)
fake_new_batch.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{DAILY_DROPS_PATH}/batch_fake_new_arrival"
)

# run ingestion again
query3 = (
    bronze_stream.writeStream.format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/data")
    .trigger(availableNow=True)
    .start(BRONZE_TABLE_PATH)
)
query3.awaitTermination()

bronze_df_after_new_file = spark.read.format("delta").load(BRONZE_TABLE_PATH)
print(f"Row count after adding ONE new file: {bronze_df_after_new_file.count()}")

In [0]:
#BRONZE TABLE actual metadata columns
bronze_df.groupBy("_source_file").count().orderBy(F.desc("count")).show(20, truncate=False)

i simulated incremental file arrival by splitting the static olist dataset into weekly batches, since auto loader's core value( only processing new files) cannot be demonstrated agiainst a single static file load. i verified idempotency by re-running ingestion with no new files (zero rows added) and by adding a single new file( exactly that files rows were added, nothing else reprocessed)"
